TP 4-5-6 Casse brique
 
Le but de ce TP est de développer votre propre version du jeu classique Casse-brique (Breakout, 1976).
Le développement se fera en Python avec le module Tkinter pour l’interface graphique. Vous avez 3
séances de 4 heures, travail en binôme, et programmation orientée objet.
La notation portera autant sur la qualité du code (découpage en classes, lisibilité, commentaires
pertinents, utilisation de Git) que sur le résultat final.
Le but du jeu est de détruire tous les briques disposées en haut du canevas, à l’aide d’une balle
rebondissante renvoyée par une raquette contrôlée par le joueur. La partie est perdue si la balle tombe
en bas de l’écran.

Donc : POO, découpage en classe, bonnes pratiques, Git.





TO DO : 

In [ ]:
"""Function file TP4
Casse brique
Author: William GIRARDET
Date: October 8-22th 2026
ToDo: 
    Stocker les vies sous forme de pile
    Stocker les niveaux sous forme de file
    Matrice de briques correspondant à chaque niveau 
        0 --> espace vide
        1 --> brique classique 
        2 et + --> brique renforcée
        -1 --> pouvoir 
        -2 --> brique spéciale (effet)

    Menu : ouvrir une nouvelle fenêtre 
    - Resume
    - Restart current level / game
    - Level select
    - Quit

    Revoir le déplacement 

    "Each frame" :
    - check bar displacement 
    - check ball position 

    Create multiple zones in playing field:
    - bar / dead zone (could be separate or not)
    - empty zone (middle space where nothing happens)
    - game zone (main outline of where the bricks are, actually the brick matrix)
"""


import random   
import tkinter as tk

class Game:


class brick:
    def __init__(self, x, y, type):
        self.x = x
        self.y = y
        self.type = type
        self.is_destroyed = False

    def destroy(self):
        self.is_destroyed = True

    def is_destroyed(self):
        return self.is_destroyed



class Bille : 
    

Code Claude ci dessous :

In [ ]:

import random
import tkinter as tk
 
TAILLE = 500          # côté de l'espace de jeu (carré)
DELAI = 16            # ms entre deux images (~60 images/s)
 
 
class Plateforme:
    def __init__(self, canvas):
        self.canvas = canvas
        self.largeur = 90
        self.hauteur = 12
        self.vitesse = 8
        self.direction = 0  # -1 gauche, 0 immobile, +1 droite
        x = (TAILLE - self.largeur) / 2
        y = TAILLE - 30
        self.id = canvas.create_rectangle(x, y, x + self.largeur, y + self.hauteur,
                                          fill="white", outline="")
 
    def deplacer(self):
        x1, _, x2, _ = self.canvas.coords(self.id)
        dx = self.direction * self.vitesse
        # empêche de sortir de l'espace de jeu
        if x1 + dx < 0:
            dx = -x1
        elif x2 + dx > TAILLE:
            dx = TAILLE - x2
        self.canvas.move(self.id, dx, 0)
 
    def coords(self):
        return self.canvas.coords(self.id)
 
 
class Bille:
    def __init__(self, canvas):
        self.canvas = canvas
        self.rayon = 8
        self.id = canvas.create_oval(0, 0, 2 * self.rayon, 2 * self.rayon,
                                     fill="deepskyblue", outline="")
        self.reinitialiser()
 
    def reinitialiser(self):
        x = TAILLE / 2
        y = TAILLE / 2
        self.canvas.coords(self.id, x - self.rayon, y - self.rayon,
                           x + self.rayon, y + self.rayon)
        self.dx = random.choice([-4, 4])
        self.dy = -4
 
    def deplacer(self, plateforme):
        """Déplace la bille. Retourne False si elle tombe en bas."""
        self.canvas.move(self.id, self.dx, self.dy)
        x1, y1, x2, y2 = self.canvas.coords(self.id)
 
        # Rebonds sur les murs gauche, droite et haut
        if x1 <= 0:
            self.dx = abs(self.dx)
        elif x2 >= TAILLE:
            self.dx = -abs(self.dx)
        if y1 <= 0:
            self.dy = abs(self.dy)
 
        # Rebond sur la plateforme
        px1, py1, px2, py2 = plateforme.coords()
        if self.dy > 0 and y2 >= py1 and y1 <= py2 and x2 >= px1 and x1 <= px2:
            self.dy = -abs(self.dy)
            # l'angle dépend de l'endroit où la bille touche la plateforme
            centre_bille = (x1 + x2) / 2
            position = (centre_bille - (px1 + px2) / 2) / ((px2 - px1) / 2)  # -1 à 1
            self.dx = position * 6
            # petite accélération à chaque rebond
            self.dy = max(-9, self.dy * 1.03)
            return True, True
 
        # Bille perdue
        if y1 >= TAILLE:
            return False, False
        return True, False
 
 
class Jeu(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Bille et plateforme")
        self.resizable(False, False)
 
        self.canvas = tk.Canvas(self, width=TAILLE, height=TAILLE,
                                bg="black", highlightthickness=0)
        self.canvas.pack()
 
        self.plateforme = Plateforme(self.canvas)
        self.bille = Bille(self.canvas)
 
        self.score = 0
        self.en_cours = True
        self.texte_score = self.canvas.create_text(10, 10, anchor="nw", fill="white",
                                                   font=("Arial", 14), text="Rebonds : 0")
        self.texte_fin = None
 
        # Contrôles : on mémorise les touches enfoncées pour un mouvement fluide
        self.touches = set()
        self.bind("<KeyPress-Left>", lambda e: self._touche(e, True))
        self.bind("<KeyPress-Right>", lambda e: self._touche(e, True))
        self.bind("<KeyRelease-Left>", lambda e: self._touche(e, False))
        self.bind("<KeyRelease-Right>", lambda e: self._touche(e, False))
        self.bind("<space>", lambda e: self.rejouer())
 
        self.boucle()
 
    def _touche(self, event, enfoncee):
        if enfoncee:
            self.touches.add(event.keysym)
        else:
            self.touches.discard(event.keysym)
        gauche = "Left" in self.touches
        droite = "Right" in self.touches
        self.plateforme.direction = (1 if droite else 0) - (1 if gauche else 0)
 
    def boucle(self):
        if self.en_cours:
            self.plateforme.deplacer()
            vivant, rebond = self.bille.deplacer(self.plateforme)
            if rebond:
                self.score += 1
                self.canvas.itemconfig(self.texte_score, text=f"Rebonds : {self.score}")
            if not vivant:
                self.fin_de_partie()
        self.after(DELAI, self.boucle)
 
    def fin_de_partie(self):
        self.en_cours = False
        self.texte_fin = self.canvas.create_text(
            TAILLE / 2, TAILLE / 2, fill="red", font=("Arial", 22, "bold"), justify="center",
            text=f"Perdu !\nScore : {self.score}\n\nEspace pour rejouer")
 
    def rejouer(self):
        if self.en_cours:
            return
        if self.texte_fin:
            self.canvas.delete(self.texte_fin)
            self.texte_fin = None
        self.score = 0
        self.canvas.itemconfig(self.texte_score, text="Rebonds : 0")
        self.bille.reinitialiser()
        self.en_cours = True
 
 
if __name__ == "__main__":
    Jeu().mainloop()
 


test quoicoutest

code broken qui fonctionne pixel : 

In [ ]:
"""
CASSE-BRIQUE - Programmation orientée objet (Python / Tkinter)

Structures de données implémentées "à la main" (sans list/deque de Python) :
  - LISTE  (Liste)  : liste chaînée contenant les briques encore présentes.
  - FILE   (File)   : FIFO (premier entré, premier sorti) -> les niveaux à jouer.
  - PILE   (Pile)   : LIFO (dernier entré, premier sorti) -> les vies du joueur.

Commandes : flèches gauche/droite = raquette, Espace = lancer la balle, P = pause.
"""

import math
import random
import tkinter as tk
from tkinter import messagebox

LARGEUR, HAUTEUR = 640, 480
COLONNES = 10
COULEURS = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#3498db", "#9b59b6"]


# ======================================================================
#  STRUCTURES DE DONNÉES
# ======================================================================
class Noeud:
    """Maillon de base : une valeur + un lien vers le maillon suivant."""

    def __init__(self, valeur):
        self.valeur = valeur
        self.suivant = None


class Liste:
    """LISTE CHAÎNÉE : on peut parcourir tous les éléments et en supprimer
    n'importe lequel (ici : une brique touchée, quelle que soit sa position)."""

    def __init__(self):
        self.tete = None
        self.taille = 0

    def ajouter(self, valeur):
        noeud = Noeud(valeur)
        noeud.suivant = self.tete
        self.tete = noeud
        self.taille += 1

    def supprimer(self, valeur):
        precedent, courant = None, self.tete
        while courant is not None:
            if courant.valeur is valeur:
                if precedent is None:
                    self.tete = courant.suivant
                else:
                    precedent.suivant = courant.suivant
                self.taille -= 1
                return True
            precedent, courant = courant, courant.suivant
        return False

    def est_vide(self):
        return self.tete is None

    def __len__(self):
        return self.taille

    def __iter__(self):
        courant = self.tete
        while courant is not None:
            yield courant.valeur
            courant = courant.suivant


class File:
    """FILE (FIFO) : on enfile à la QUEUE, on défile par la TÊTE.
    Le premier niveau enfilé est le premier joué."""

    def __init__(self):
        self.tete = None
        self.queue = None
        self.taille = 0

    def enfiler(self, valeur):
        noeud = Noeud(valeur)
        if self.queue is None:
            self.tete = self.queue = noeud
        else:
            self.queue.suivant = noeud
            self.queue = noeud
        self.taille += 1

    def defiler(self):
        if self.tete is None:
            return None
        valeur = self.tete.valeur
        self.tete = self.tete.suivant
        if self.tete is None:
            self.queue = None
        self.taille -= 1
        return valeur

    def est_vide(self):
        return self.tete is None

    def __len__(self):
        return self.taille

    def __iter__(self):
        courant = self.tete
        while courant is not None:
            yield courant.valeur
            courant = courant.suivant


class Pile:
    """PILE (LIFO) : on empile et on dépile toujours par le SOMMET.
    La dernière vie empilée est la première perdue."""

    def __init__(self):
        self._sommet = None
        self.taille = 0

    def empiler(self, valeur):
        noeud = Noeud(valeur)
        noeud.suivant = self._sommet
        self._sommet = noeud
        self.taille += 1

    def depiler(self):
        if self._sommet is None:
            return None
        valeur = self._sommet.valeur
        self._sommet = self._sommet.suivant
        self.taille -= 1
        return valeur

    def sommet(self):
        return None if self._sommet is None else self._sommet.valeur

    def est_vide(self):
        return self._sommet is None

    def __len__(self):
        return self.taille

    def __iter__(self):  # du sommet vers le fond
        courant = self._sommet
        while courant is not None:
            yield courant.valeur
            courant = courant.suivant


# ======================================================================
#  ÉLÉMENTS DU JEU
# ======================================================================
class Brique:
    def __init__(self, canvas, x, y, largeur, hauteur, couleur, points):
        self.x, self.y, self.l, self.h = x, y, largeur, hauteur
        self.points = points
        self.canvas = canvas
        self.id = canvas.create_rectangle(x, y, x + largeur, y + hauteur,
                                          fill=couleur, outline="#222")

    def detruire(self):
        self.canvas.delete(self.id)


class Raquette:
    def __init__(self, canvas):
        self.canvas = canvas
        self.l, self.h, self.vitesse = 90, 12, 8
        self.x = (LARGEUR - self.l) / 2
        self.y = HAUTEUR - 35
        self.id = canvas.create_rectangle(0, 0, 0, 0, fill="#ecf0f1", outline="")
        self.dessiner()

    def deplacer(self, touches):
        if "Left" in touches:
            self.x -= self.vitesse
        if "Right" in touches:
            self.x += self.vitesse
        self.x = max(0, min(LARGEUR - self.l, self.x))
        self.dessiner()

    def dessiner(self):
        self.canvas.coords(self.id, self.x, self.y, self.x + self.l, self.y + self.h)


class Balle:
    def __init__(self, canvas, vitesse):
        self.canvas = canvas
        self.r = 7
        self.v = vitesse
        self.x = self.y = 0
        self.dx = self.dy = 0
        self.id = canvas.create_oval(0, 0, 0, 0, fill="#f39c12", outline="")

    def poser_sur(self, raquette):
        """Place la balle sur la raquette et choisit une direction de départ."""
        self.x = raquette.x + raquette.l / 2
        self.y = raquette.y - self.r - 1
        self.dx = random.choice([-1, 1]) * self.v * 0.5
        self.dy = -math.sqrt(self.v ** 2 - self.dx ** 2)
        self.dessiner()

    def deplacer(self):
        self.x += self.dx
        self.y += self.dy
        self.dessiner()

    def dessiner(self):
        r = self.r
        self.canvas.coords(self.id, self.x - r, self.y - r, self.x + r, self.y + r)

    def rebond_murs(self):
        if self.x - self.r <= 0:
            self.x, self.dx = self.r, abs(self.dx)
        elif self.x + self.r >= LARGEUR:
            self.x, self.dx = LARGEUR - self.r, -abs(self.dx)
        if self.y - self.r <= 0:
            self.y, self.dy = self.r, abs(self.dy)

    def rebond_raquette(self, raq):
        bas = self.y + self.r
        if (self.dy > 0 and raq.y <= bas <= raq.y + raq.h + self.v
                and raq.x - self.r <= self.x <= raq.x + raq.l + self.r):
            decalage = (self.x - (raq.x + raq.l / 2)) / (raq.l / 2)  # -1 .. +1
            decalage = max(-1, min(1, decalage))
            self.dx = decalage * self.v * 0.8
            self.dy = -math.sqrt(self.v ** 2 - self.dx ** 2)
            self.y = raq.y - self.r - 1
            return True
        return False

    def rebond_brique(self, b):
        g, d = self.x - self.r, self.x + self.r
        h, bas = self.y - self.r, self.y + self.r
        if d < b.x or g > b.x + b.l or bas < b.y or h > b.y + b.h:
            return False
        # côté où la pénétration est la plus faible = côté touché
        ov_x = min(d - b.x, b.x + b.l - g)
        ov_y = min(bas - b.y, b.y + b.h - h)
        if ov_x < ov_y:
            self.dx = -self.dx
        else:
            self.dy = -self.dy
        return True


# Description des niveaux (chaque niveau sera ENFILÉ dans la File)
NIVEAUX = [
    {"nom": "Niveau 1", "lignes": 4, "motif": lambda i, j: True},
    {"nom": "Niveau 2", "lignes": 5, "motif": lambda i, j: (i + j) % 2 == 0},
    {"nom": "Niveau 3", "lignes": 6, "motif": lambda i, j: j not in (4, 5)},
]


# ======================================================================
#  LOGIQUE DU JEU
# ======================================================================
class Jeu:
    def __init__(self, canvas, vitesse, maj_score):
        self.canvas = canvas
        self.maj_score = maj_score
        self.score = 0
        self.touches = set()
        self.pause = False
        self.etat = "en_cours"       # "en_cours" | "attente" | "fini"
        self.job = None

        self.briques = Liste()       # LISTE : briques présentes à l'écran
        self.niveaux = File()        # FILE  : niveaux restant à jouer
        self.vies = Pile()           # PILE  : vies restantes

        for niveau in NIVEAUX:
            self.niveaux.enfiler(niveau)
        for _ in range(3):
            self.vies.empiler("♥")

        self.niveau_courant = None
        self.raquette = Raquette(canvas)
        self.balle = Balle(canvas, vitesse)

        self.charger_niveau_suivant()
        self.balle.poser_sur(self.raquette)
        self.maj_score(self.score)
        self.afficher_hud()
        self.boucle()

    # --- niveaux / affichage ------------------------------------------
    def charger_niveau_suivant(self):
        if self.niveaux.est_vide():
            return False
        niveau = self.niveaux.defiler()          # FILE : on sort le plus ancien
        self.niveau_courant = niveau
        larg_case = (LARGEUR - 40) / COLONNES
        for i in range(niveau["lignes"]):
            for j in range(COLONNES):
                if niveau["motif"](i, j):
                    self.briques.ajouter(Brique(          # LISTE : on ajoute
                        self.canvas, 20 + j * larg_case + 2, 50 + i * 24,
                        larg_case - 4, 20, COULEURS[i % len(COULEURS)],
                        (niveau["lignes"] - i) * 10))
        return True

    def afficher_hud(self):
        self.canvas.delete("hud")
        nom = self.niveau_courant["nom"] if self.niveau_courant else ""
        self.canvas.create_text(10, 15, anchor="w", fill="white", tag="hud",
                                font=("Arial", 12, "bold"), text=nom)
        self.canvas.create_text(LARGEUR - 10, 15, anchor="e", fill="#e74c3c",
                                tag="hud", font=("Arial", 14),
                                text="".join(self.vies))   # parcours de la PILE

    def message(self, texte):
        self.effacer_message()
        self.canvas.create_text(LARGEUR / 2, HAUTEUR / 2, fill="white", tag="msg",
                                font=("Arial", 18, "bold"), text=texte, justify="center")

    def effacer_message(self):
        self.canvas.delete("msg")

    # --- commandes ----------------------------------------------------
    def touche_pressee(self, touche):
        if touche in ("Left", "Right"):
            self.touches.add(touche)
        elif touche == "space" and self.etat == "attente":
            self.etat = "en_cours"
            self.effacer_message()
        elif touche in ("p", "P") and self.etat != "fini":
            self.pause = not self.pause
            if self.pause:
                self.message("PAUSE")
            else:
                self.effacer_message()

    def touche_relachee(self, touche):
        self.touches.discard(touche)

    def basculer_pause(self):
        self.touche_pressee("p")

    # --- boucle principale -------------------------------------------
    def boucle(self):
        if not self.pause and self.etat != "fini":
            self.raquette.deplacer(self.touches)
            if self.etat == "en_cours":
                self.mise_a_jour()
            else:  # "attente" : la balle suit la raquette
                self.balle.x = self.raquette.x + self.raquette.l / 2
                self.balle.dessiner()
        self.job = self.canvas.after(16, self.boucle)

    def mise_a_jour(self):
        balle = self.balle
        balle.deplacer()
        balle.rebond_murs()
        balle.rebond_raquette(self.raquette)

        touchee = None
        for brique in self.briques:              # parcours de la LISTE
            if balle.rebond_brique(brique):
                touchee = brique
                break
        if touchee is not None:
            self.briques.supprimer(touchee)      # suppression dans la LISTE
            touchee.detruire()
            self.score += touchee.points
            self.maj_score(self.score)
            if self.briques.est_vide():
                self.niveau_termine()
                return

        if balle.y - balle.r > HAUTEUR:
            self.perdre_vie()

    def perdre_vie(self):
        self.vies.depiler()                      # PILE : on retire la dernière vie
        self.afficher_hud()
        if self.vies.est_vide():
            self.fin("GAME OVER")
        else:
            self.balle.poser_sur(self.raquette)
            self.etat = "attente"
            self.message("Vie perdue !\nEspace pour relancer")

    def niveau_termine(self):
        if self.charger_niveau_suivant():        # FILE : niveau suivant
            self.afficher_hud()
            self.balle.poser_sur(self.raquette)
            self.etat = "attente"
            self.message(f"{self.niveau_courant['nom']}\nEspace pour lancer")
        else:
            self.fin("VICTOIRE !")

    def fin(self, texte):
        self.etat = "fini"
        self.message(f"{texte}\nScore : {self.score}")

    def arreter(self):
        if self.job is not None:
            self.canvas.after_cancel(self.job)
            self.job = None

    def resume_structures(self):
        noms = [n["nom"] for n in self.niveaux]
        return (f"LISTE (briques restantes) : {len(self.briques)} élément(s)\n\n"
                f"FILE (niveaux à venir, prochain en tête) : "
                f"{noms if noms else 'vide'}\n\n"
                f"PILE (vies, sommet en premier) : {''.join(self.vies) or 'vide'}")


# ======================================================================
#  INTERFACE GRAPHIQUE
# ======================================================================
class Application:
    def __init__(self, racine):
        self.racine = racine
        self.jeu = None
        racine.title("Casse-brique")
        racine.resizable(False, False)
        racine.protocol("WM_DELETE_WINDOW", self.quitter)

        self.var_score = tk.StringVar(value="Score : 0")
        self.var_vitesse = tk.IntVar(value=5)

        self.construire_menu()
        self.construire_widgets()

        racine.bind("<KeyPress>", lambda e: self.jeu and self.jeu.touche_pressee(e.keysym))
        racine.bind("<KeyRelease>", lambda e: self.jeu and self.jeu.touche_relachee(e.keysym))

        self.dessiner_accueil()

    def construire_menu(self):
        barre = tk.Menu(self.racine)

        m_partie = tk.Menu(barre, tearoff=0)
        m_partie.add_command(label="Nouvelle partie", command=self.nouvelle_partie)
        m_partie.add_command(label="Pause / Reprendre (P)", command=self.pause)
        m_partie.add_separator()
        m_partie.add_command(label="Quitter", command=self.quitter)
        barre.add_cascade(label="Partie", menu=m_partie)

        m_options = tk.Menu(barre, tearoff=0)
        for nom, v in (("Facile", 4), ("Normal", 5), ("Difficile", 7)):
            m_options.add_radiobutton(label=f"Difficulté : {nom}",
                                      variable=self.var_vitesse, value=v)
        barre.add_cascade(label="Options", menu=m_options)

        m_aide = tk.Menu(barre, tearoff=0)
        m_aide.add_command(label="Voir liste / file / pile", command=self.voir_structures)
        m_aide.add_command(label="À propos", command=lambda: messagebox.showinfo(
            "À propos", "Casse-brique en POO\nPython + Tkinter"))
        barre.add_cascade(label="Aide", menu=m_aide)

        self.racine.config(menu=barre)

    def construire_widgets(self):
        self.canvas = tk.Canvas(self.racine, width=LARGEUR, height=HAUTEUR,
                                bg="#1b2631", highlightthickness=0)
        self.canvas.pack()

        bas = tk.Frame(self.racine)
        bas.pack(fill="x", pady=6, padx=8)

        # Zone de texte affichant le score (lecture seule)
        tk.Entry(bas, textvariable=self.var_score, state="readonly", width=16,
                 font=("Arial", 12, "bold"), justify="center").pack(side="left")

        tk.Button(bas, text="Quitter", command=self.quitter,
                  takefocus=False).pack(side="right")
        tk.Button(bas, text="Démarrer", command=self.nouvelle_partie,
                  takefocus=False).pack(side="right", padx=6)

    def dessiner_accueil(self):
        self.canvas.delete("all")
        self.canvas.create_text(LARGEUR / 2, HAUTEUR / 2, fill="white",
                                font=("Arial", 16, "bold"), justify="center",
                                text="CASSE-BRIQUE\n\nCliquez sur « Démarrer »\n"
                                     "← → : raquette   Espace : lancer   P : pause")

    # --- actions --------------------------------------------------------
    def nouvelle_partie(self):
        if self.jeu:
            self.jeu.arreter()
        self.canvas.delete("all")
        self.jeu = Jeu(self.canvas, self.var_vitesse.get(),
                       lambda s: self.var_score.set(f"Score : {s}"))
        self.canvas.focus_set()

    def pause(self):
        if self.jeu:
            self.jeu.basculer_pause()

    def voir_structures(self):
        if self.jeu:
            messagebox.showinfo("Structures de données", self.jeu.resume_structures())
        else:
            messagebox.showinfo("Structures de données", "Lancez d'abord une partie.")

    def quitter(self):
        if messagebox.askyesno("Quitter", "Voulez-vous vraiment quitter ?"):
            if self.jeu:
                self.jeu.arreter()    # annule la boucle after() proprement
            self.racine.destroy()


if __name__ == "__main__":
    fenetre = tk.Tk()
    Application(fenetre)
    fenetre.mainloop()